# Zenith on Colab: tests, LR sweep, held-out eval, resume

Before running, upload to Google Drive under `MyDrive/zenith/`:
- `zenith_code.zip` (the packaged repo)
- `train.jsonl` and `heldout.jsonl` (in `zenith_data/`)
- `tokenizer32k/` folder
- `ckpt_step9000/` folder (optional, only for the resume cell)

Use a GPU runtime (Runtime > Change runtime type). T4 has no bf16, so the code falls back to fp32 there; L4 or A100 use bf16.

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/zenith'
!mkdir -p /content/zenith && cd /content/zenith && unzip -qo {DRIVE}/zenith_code.zip
!ls /content/zenith

In [ ]:
!pip install -q transformers datasets tokenizers pytest pytest-cov

In [ ]:
%cd /content/zenith
!python -m pytest -q --cov=zenith --cov-report=term-missing 2>&1 | tail -15

In [ ]:
%cd /content/zenith
import os, subprocess
DATA = f'{DRIVE}/zenith_data'
SWEEP = f'{DRIVE}/sweep'
for lr in ['5e-4', '2e-4', '1e-4']:
    out = f'{SWEEP}/lr{lr}'
    if os.path.exists(f'{out}/eval.log'):
        continue
    os.makedirs(out, exist_ok=True)
    subprocess.run(['python', '-m', 'zenith.train', '--config', 'configs/target/config.json',
        '--tokenizer', f'{DATA}/tokenizer32k', '--data', f'{DATA}/train.jsonl', '--output_dir', f'{out}/ckpt',
        '--seq_len', '512', '--batch_size', '2', '--max_steps', '300', '--warmup_steps', '30', '--lr', lr,
        '--save_steps', '100000', '--log_steps', '25', '--compile_scan'],
        stdout=open(f'{out}/train.log', 'w'), stderr=subprocess.STDOUT, check=True)
    subprocess.run(['python', 'scripts/eval_ckpt.py', '--checkpoints', f'{out}/ckpt', '--tokenizer', f'{DATA}/tokenizer32k',
        '--heldout', f'{DATA}/heldout.jsonl', '--max_windows', '128'],
        stdout=open(f'{out}/eval.log', 'w'), stderr=subprocess.STDOUT, check=True)
    print(lr, open(f'{out}/eval.log').read().strip().splitlines()[-1])

In [ ]:
%cd /content/zenith
!python scripts/eval_ckpt.py --checkpoints {DRIVE}/ckpt_step9000 --tokenizer {DRIVE}/zenith_data/tokenizer32k --heldout {DRIVE}/zenith_data/heldout.jsonl --max_windows 128 2>&1 | grep heldout_loss

## Optional: resume the main run from the step-9000 snapshot

Copies the snapshot (which includes `train_state.pt`) into a run folder, so the trainer resumes at step 9000. Checkpoints go to Drive every 1000 steps. Re-run this cell after a session timeout to continue from the last save.

Set `LR` below before running (the sweep results decide the value).

In [ ]:
%cd /content/zenith
import shutil
LR = '5e-4'
RUN = f'{DRIVE}/run_main/ckpt'
if not os.path.exists(RUN):
    shutil.copytree(f'{DRIVE}/ckpt_step9000', RUN)
!python -m zenith.train --config configs/target/config.json --tokenizer {DRIVE}/zenith_data/tokenizer32k --data {DRIVE}/zenith_data/train.jsonl --output_dir {RUN} --seq_len 512 --batch_size 2 --max_steps 33000 --warmup_steps 1000 --lr {LR} --save_steps 1000 --log_steps 50 --compile_scan